# 2026.10.19(월) 20h~

In [1]:
import os                         # 환경변수 읽기용
from dotenv import load_dotenv    # .env 파일을 읽어 환경변수로 등록하는 함수

load_dotenv()  # .env 파일에서 OPENAI_API_KEY를 읽어 옴

# 키가 없으면 여기서 바로 멈추고 안내 메시지 출력 (뒤에서 알기 어려운 에러가 나는 것을 방지)
assert os.getenv("OPENAI_API_KEY"), ".env 파일에 OPENAI_API_KEY가 없습니다."

# 실습 전체에서 사용할 모델 (gpt-4o-mini는 2027-01-14 종료 예정, 이후에는 여기만 바꾸면 됨)
MODEL = "gpt-4o-mini"

## 다단계 체인

In [2]:
from openai import OpenAI   # OpenAI 공식 파이썬 라이브러리

client = OpenAI()   # OPENAI_API_KEY 환경변수를 자동으로 사용

# GPT를 한 번 호출하는 함수 (같은 코드를 두 번 쓰지 않도록 함수로 묶음)
#   system_content : 모델의 역할 지정 (예: 요약 전문가)
#   user_content   : 실제 요청 내용
def call_gpt(system_content: str, user_content: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_content},   # 역할
            {"role": "user", "content": user_content},       # 요청
        ],
        temperature=0.7,   # 답변의 다양성 (0이면 매번 비슷한 답)
    )
    # 응답 객체에서 답변 텍스트만 꺼내고 앞뒤 공백 제거
    return response.choices[0].message.content.strip()


# 요약할 원문 (세 줄짜리 여러 줄 문자열)
original_text = """최근 인공지능 기술은 다양한 분야에서 활용되고 있습니다.
예를 들어, 의료 분야에서는 진단 보조 시스템으로 사용되고,
교육 분야에서는 개인 맞춤형 학습에 활용되고 있습니다."""

# 1단계: 원문 → 한글 요약
summary_ko = call_gpt("당신은 한국어 텍스트를 요약하는 전문가입니다.",
                      f"아래 한국어 텍스트를 간단히 요약해 주세요:\n\n{original_text}")

# 2단계: 한글 요약 → 영어 번역 (1단계 결과를 직접 넘겨 줌)
summary_en = call_gpt("당신은 한국어를 영어로 번역하는 전문 번역가입니다.",
                      f"아래 요약문을 영어로 번역해 주세요:\n\n{summary_ko}")

# 단계별 결과 출력
print("[원문]\n", original_text)
print("\n[요약]\n", summary_ko)
print("\n[번역]\n", summary_en)

[원문]
 최근 인공지능 기술은 다양한 분야에서 활용되고 있습니다.
예를 들어, 의료 분야에서는 진단 보조 시스템으로 사용되고,
교육 분야에서는 개인 맞춤형 학습에 활용되고 있습니다.

[요약]
 최근 인공지능 기술은 의료와 교육 분야에서 진단 보조 시스템과 개인 맞춤형 학습에 활용되고 있습니다.

[번역]
 Recently, artificial intelligence technologies have been applied in the fields of healthcare and education for diagnostic support systems and personalized learning.


In [4]:
from langchain_openai import ChatOpenAI                  # OpenAI 채팅 모델 (LangChain용)
from langchain_core.prompts import PromptTemplate        # 빈칸({변수})이 있는 프롬프트 틀
from langchain_core.output_parsers import StrOutputParser  # 모델 응답에서 텍스트만 꺼내는 부품

llm = ChatOpenAI(model=MODEL, temperature=0.7)

# 요약용 프롬프트: {input_text} 자리에 원문이 들어감
summarize_prompt = PromptTemplate.from_template(
    "당신은 한국어 텍스트를 요약하는 전문가입니다.\n"
    "아래 한국어 텍스트를 간단히 요약해 주세요:\n\n{input_text}"
)
# 번역용 프롬프트: {summary_text} 자리에 요약문이 들어감
translate_prompt = PromptTemplate.from_template(
    "당신은 한국어를 영어로 번역하는 전문 번역가입니다.\n"
    "아래 한국어를 영어로 번역해 주세요:\n\n{summary_text}" # 또는 '요약문을'
)

# 1단계 체인: 원문 → 한글 요약(문자열)
summarize_chain = summarize_prompt | llm | StrOutputParser()

# 2단계 연결: 요약 결과를 summary_text 자리에 넣어 번역 체인으로 전달
#   {"summary_text": summarize_chain} : 1단계 체인을 실행한 결과를 summary_text라는 이름으로 묶어 줌
#   이 이름이 translate_prompt의 {summary_text}와 같아야 연결됨
full_chain = {"summary_text": summarize_chain} | translate_prompt | llm | StrOutputParser()

# 처음 입력은 1단계 프롬프트의 변수 이름(input_text)으로 넘김
result = full_chain.invoke({"input_text": original_text})
print("[최종 결과: 한글 요약 → 영어 번역]")
print(result)

[최종 결과: 한글 요약 → 영어 번역]
Artificial intelligence technology is being used as a diagnostic support system in the medical field and for personalized learning in the education sector.


## 대화 메모리

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory   # 대화 기록을 메모리에 저장하는 클래스

# 채팅용 프롬프트: system → 이전 대화 → 이번 질문 순서로 모델에 전달
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절하고 간결하게 답하는 AI 어시스턴트입니다."),
    MessagesPlaceholder("history"),      # 이전 대화가 들어갈 자리
    ("user", "{input}"),                 # 이번에 사용자가 입력한 질문
])
chat_chain = chat_prompt | llm | StrOutputParser()

history = InMemoryChatMessageHistory()   # 대화 기록 저장소

# 질문 하나를 처리하는 함수
def chat(user_input: str) -> str:
    # 1) 지금까지의 대화 기록(history.messages)과 새 질문을 함께 모델에 전달
    answer = chat_chain.invoke({"input": user_input, "history": history.messages})
    # 2) 이번 질문과 답변을 기록에 추가 (다음 질문 때 함께 전달됨)
    history.add_user_message(user_input)
    history.add_ai_message(answer)
    return answer


print(chat("제 이름은 김철수입니다."))
print(chat("제 이름이 뭐라고 했죠?"))     # 기록 덕분에 이름을 기억해서 답함
print("\n저장된 메시지 수:", len(history.messages))
print(f"지금까지의 대화 기록 = {history.messages}")

안녕하세요, 김철수님! 어떻게 도와드릴까요?
당신의 이름은 김철수님입니다.

저장된 메시지 수: 4
지금까지의 대화 기록 = [HumanMessage(content='제 이름은 김철수입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 김철수님! 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='제 이름이 뭐라고 했죠?', additional_kwargs={}, response_metadata={}), AIMessage(content='당신의 이름은 김철수님입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
